In [1]:
import pandas as pd

df = pd.read_excel("/content/RealAndSynthetic.xlsx")

df.head()

,review_text,issue
0,I purchased these sheets based on positive Wir...,Material_Quality
1,Picked up a full sheet set from the store. Pac...,Size_Fit
2,Did not like how they came out of the washer a...,Comfort_Softness
3,"I’ve washed these twice, using detergent and l...",Chemical_Odor
4,Terrible chemical smell. Possibly formaldehyde...,Chemical_Odor


In [2]:
df.sample(10)

,review_text,issue
771,This doesn’t feel like it’s 100% cotton even t...,Material_Quality
144,These sheets feel like good quality and are fa...,Design_Issue
1073,These sheets are neither crisp like percale or...,Comfort_Softness
1441,Rough material. Washed them otherwise I would ...,Comfort_Softness
3870,The delivery was rescheduled to a day when I w...,Delivery_Delay
5313,I am thoroughly disappointed with the unhelpfu...,Customer_Service
4110,The item was damaged because the box was crush...,Shipping_Damage
3805,"The control panel is on the back, making it ha...",Design_Issue
4831,There is a dent in the box that exactly matche...,Shipping_Damage
1893,The quality of these have went down so much. T...,Comfort_Softness


In [3]:
df.isnull().sum()

,0
review_text,0
issue,1


In [5]:
df.shape

(5549, 2)

In [6]:
df = df.dropna()

In [7]:
df.shape

(5548, 2)

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["issue"])

In [12]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [13]:
train_df.shape, test_df.shape

((4438, 3), (1110, 3))

In [14]:
train_df

,review_text,issue,label
4592,There was absolutely zero padding inside the s...,Packaging,11
446,"Unfortunately, I don't know how the product is...",Missing_Parts,9
88,These are nice and I chose for the natural fib...,Chemical_Odor,0
4670,The outer carton looked like it had been dropp...,Packaging,11
4212,I was charged a 're-stocking and handling' fee...,Return_Refund,14
...,...,...,...
2378,"Liked the material. However, the ottom sheet i...",Size_Fit,17
3142,These sheets pill so badly that after washing ...,Pilling,12
1454,This is my second set of these sheets. I have ...,Size_Fit,17
1662,I didn’t look at other reviews before buying a...,Wrinkle_Resistance,18


In [15]:
pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [17]:
def tokenize(batch):
    return tokenizer(
        batch["review_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [18]:
# coverting dataset HF Dataset
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

In [19]:
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/4438 [00:00<?, ? examples/s]

Map:   0%|          | 0/1110 [00:00<?, ? examples/s]

In [20]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=20
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01
)

In [22]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted"
        )
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [23]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.413336,0.666667,0.689594,0.666667,0.634074
2,1.686548,0.911575,0.762162,0.767056,0.762162,0.753844
3,1.686548,0.737310,0.798198,0.804036,0.798198,0.796980
4,0.645156,0.676053,0.805405,0.806361,0.805405,0.801649
5,0.645156,0.656154,0.807207,0.805932,0.807207,0.803266


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1390, training_loss=0.9502954826080542, metrics={'train_runtime': 321.2764, 'train_samples_per_second': 69.068, 'train_steps_per_second': 4.326, 'total_flos': 735098788300800.0, 'train_loss': 0.9502954826080542, 'epoch': 5.0})

In [25]:
from sklearn.metrics import (
    classification_report
)

predictions = trainer.predict(
    test_ds
)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = predictions.label_ids

print(
    classification_report(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.83      0.83      0.83        41
           1       0.83      0.79      0.81        38
           2       0.89      0.92      0.91       212
           3       0.94      0.82      0.88        40
           4       0.67      0.62      0.64        39
           5       0.95      1.00      0.98        41
           6       0.53      0.28      0.37        32
           7       0.89      0.84      0.86        80
           8       0.55      0.54      0.54       110
           9       0.72      0.79      0.75        29
          10       0.75      0.58      0.65        26
          11       0.83      0.95      0.88        40
          12       0.72      0.86      0.78        57
          13       0.94      0.72      0.82        40
          14       0.87      0.94      0.90        78
          15       0.97      0.91      0.94        43
          16       0.87      0.85      0.86        40
          17       0.72    

In [26]:
trainer.save_model(
    "issue_classifier"
)

tokenizer.save_pretrained(
    "issue_classifier"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('issue_classifier/tokenizer_config.json', 'issue_classifier/tokenizer.json')

In [27]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

tokenizer = AutoTokenizer.from_pretrained(
    "issue_classifier"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "issue_classifier"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [28]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

review = """
The package arrived crushed and the box was torn.
"""

result = classifier(review)

print(result)

[{'label': 'LABEL_15', 'score': 0.6441042423248291}]


In [35]:
print(le.classes_)

['Chemical_Odor' 'Color_Appearance' 'Comfort_Softness' 'Customer_Service'
 'Defect' 'Delivery_Delay' 'Design_Issue' 'Durability' 'Material_Quality'
 'Missing_Parts' 'Other' 'Packaging' 'Pilling' 'Positive' 'Return_Refund'
 'Shipping_Damage' 'Shrinkage' 'Size_Fit' 'Wrinkle_Resistance'
 'Wrong_Item']


In [32]:
import torch

def predict_issue_with_confidence(text):

    device = model.device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )

    pred_idx = probs.argmax().item()

    confidence = probs.max().item()

    issue = le.inverse_transform(
        [pred_idx]
    )[0]

    return issue, confidence

In [38]:
predict_issue_with_confidence(
    "They sent the wrong item sent house shoes instead of pillowcases"
)

('Wrong_Item', 0.7261860370635986)

In [31]:
print(model.device)

cuda:0
